# Sliding-window inference, step by step

The network sees 128³ windows, but a brain volume is larger. This notebook rebuilds
what `sliding_window_inference` does for one case and shows:

1. how the volume is covered by overlapping windows (2D and 3D),
2. the Gaussian weights used to blend overlapping windows,
3. the model output for a single window,
4. the same inference written as an explicit loop, checked against MONAI,
5. how many windows cover each voxel,
6. how the final label of a single voxel is decided.

The case comes from the **validation** split (the test split stays untouched).
Slices are shown with x (left → right) horizontally and y (posterior → anterior) vertically.

In [30]:
import random
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import torch
import torch.nn.functional as F
from monai.data.utils import compute_importance_map, dense_patch_slices
from plotly.subplots import make_subplots
from scipy.ndimage import gaussian_filter
from skimage.measure import marching_cubes

REPO_ROOT = Path.cwd().resolve()
while not (REPO_ROOT / "src").is_dir() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))

from src.dataset.dataset import get_datasets
from src.dataset.paths import get_output_dir
from src.dataset.transforms import REGIONS
from src.evaluation import predict_logits
from src.model import build_unet

RUN_NAME = "unet_baseline"
CHECKPOINT = "best"
CASE_ID = None      # None = random validation case
SEED = 0
OVERLAP = 0.5       # same as evaluation
STATIC = {"staticPlot": True}

DEVICE = torch.device("cuda", 0) if torch.cuda.is_available() else torch.device("cpu")
AMP = DEVICE.type == "cuda"

LABEL_NAMES = {0: "background", 1: "NETC", 2: "SNFH", 3: "ET", 4: "RC"}

## 1. Model and case

In [31]:
state = torch.load(get_output_dir() / "runs" / RUN_NAME / "checkpoints" / f"{CHECKPOINT}.pt",
                   map_location=DEVICE, weights_only=False)
include_rc = not state["config"].get("no_rc", False)
ROI = (state["config"]["patch"],) * 3

model = build_unet(include_rc=include_rc).to(DEVICE)
model.load_state_dict(state["model"])
model.eval()

dataset = get_datasets(patch_size=ROI, include_rc=include_rc)["val"]
case_ids = [r["case_id"] for r in dataset.data]
idx = case_ids.index(CASE_ID) if CASE_ID else random.Random(SEED).randrange(len(case_ids))
sample = dataset[idx]
case_id, image, label = sample["case_id"], sample["image"], sample["label"]
regions = list(REGIONS[: label.shape[0]])
print(f"{case_id}: volume {tuple(image.shape[1:])}, window {ROI}, overlap {OVERLAP}")


def regions_to_labels(r: np.ndarray) -> np.ndarray:
    """Region masks (C, ...) -> labels 0-4, innermost region wins."""
    r = r.astype(bool)
    lab = np.zeros(r.shape[1:], dtype=np.uint8)
    if r.shape[0] > 3:
        lab[r[3]] = 4
    lab[r[2]] = 2
    lab[r[1]] = 1
    lab[r[0]] = 3
    return lab


gt_regions = label.numpy().astype(bool)
gt_lab = regions_to_labels(gt_regions)
brain = (image != 0).any(dim=0).numpy()

BraTS-GLI-02885-101: volume (143, 172, 141), window (128, 128, 128), overlap 0.5


## 2. Covering the volume with windows

Windows of 128³ are placed on a regular grid. With overlap 0.5 the step (*scan interval*)
is 64 voxels; the last window on each axis is shifted back so it ends at the volume edge.
An axis shorter than 128 is first zero-padded symmetrically to 128.
These rules are MONAI's (`_get_scan_interval`, `dense_patch_slices`); they are reproduced here.

In [32]:
size = tuple(image.shape[1:])
padded = tuple(max(s, r) for s, r in zip(size, ROI))
offset = tuple((p - s) // 2 for p, s in zip(padded, size))       # padding before each axis
interval = tuple(r if r == p else max(int(r * (1 - OVERLAP)), 1) for r, p in zip(ROI, padded))
windows = dense_patch_slices(padded, ROI, interval)               # list of (slice_x, slice_y, slice_z)

# zero-pad like MONAI (F.pad takes the last axis first)
pad = []
for s, p, o in reversed(list(zip(size, padded, offset))):
    pad += [o, p - s - o]
image_p = F.pad(image, pad)

starts = [sorted({w[a].start for w in windows}) for a in range(3)]
pd.DataFrame({
    "volume": size, "padded": padded, "window": ROI, "scan interval": interval,
    "window starts": [str(s) for s in starts], "windows on axis": [len(s) for s in starts],
}, index=["x", "y", "z"]).T

,x,y,z
volume,143,172,141
padded,143,172,141
window,128,128,128
scan interval,64,64,64
window starts,"[np.int64(0), np.int64(15)]","[np.int64(0), np.int64(44)]","[np.int64(0), np.int64(13)]"
windows on axis,2,2,2


In [33]:
print(f"{len(windows)} windows in total = " + " × ".join(str(len(s)) for s in starts))


def centroid(mask):
    idx = np.argwhere(mask)
    return tuple(int(v) for v in idx.mean(0)) if len(idx) else tuple(s // 2 for s in mask.shape)


def xy_slice(vol, z):
    """Axial slice with x horizontal, y vertical (anterior up)."""
    return vol[:, :, z].T


wt = gt_regions[2]
cz = int(wt.sum(axis=(0, 1)).argmax())            # slice with the most tumour voxels
cx, cy = (int(v) for v in np.argwhere(wt[:, :, cz]).mean(0))  # tumour centre on that slice# whole-tumour centre
colours = ["#1f77b4", "#ff7f0e", "#2ca02c", "#9467bd", "#8c564b", "#e377c2", "#17becf", "#bcbd22"]

fig = go.Figure(go.Heatmap(z=xy_slice(image[1].numpy(), cz), colorscale="gray", showscale=False,
                           hoverinfo="skip"))
fig.add_trace(go.Contour(z=xy_slice(gt_regions[2].astype(float), cz), contours=dict(start=0.5, end=0.5,
                         coloring="none"), line=dict(color="#d62728", width=2), showscale=False,
                         hoverinfo="skip", name="whole tumour (truth)"))
footprints = sorted({(w[0].start, w[1].start) for w in windows if w[2].start - offset[2] <= cz < w[2].stop - offset[2]})
for n, (x0, y0) in enumerate(footprints):
    x0, y0 = x0 - offset[0], y0 - offset[1]
    fig.add_shape(type="rect", x0=x0 - 0.5, x1=x0 + ROI[0] - 0.5, y0=y0 - 0.5, y1=y0 + ROI[1] - 0.5,
                  line=dict(color=colours[n % len(colours)], width=3, dash="dash" if n % 2 else "solid"))
fig.update_xaxes(visible=False, range=[min(-0.5, -offset[0]) - 2, max(size[0], padded[0] - offset[0]) + 2])
fig.update_yaxes(visible=False, scaleanchor="x",
                 range=[min(-0.5, -offset[1]) - 2, max(size[1], padded[1] - offset[1]) + 2])
fig.update_layout(width=650, height=650, plot_bgcolor="white", margin=dict(t=60, b=10, l=10, r=10),
                  title=dict(text=f"<b>Window footprints on slice z = {cz}</b><br>"
                                  f"{len(footprints)} x-y positions; each is repeated along z", x=0.5))
fig.show(config=STATIC)

8 windows in total = 2 × 2 × 2


### The same grid in 3D

Every box is one window. The red box contains the centre of the tumour.
Drag to rotate.

In [34]:
def mesh(mask, step=1, sigma=1.0):
    if mask.sum() < 20:
        return None
    padded_mask = np.pad(mask.astype(np.float32), 2)
    vol = gaussian_filter(padded_mask, sigma)
    if vol.max() <= 0.5:
        vol = padded_mask
    v, f, _, _ = marching_cubes(vol, level=0.5, step_size=step)
    return v - 2, f


def box_lines(x0, y0, z0, dx, dy, dz):
    """12 edges of a box as line segments separated by None."""
    c = np.array([[x0 + i * dx, y0 + j * dy, z0 + k * dz] for i in (0, 1) for j in (0, 1) for k in (0, 1)])
    edges = [(0, 1), (0, 2), (0, 4), (1, 3), (1, 5), (2, 3), (2, 6), (3, 7), (4, 5), (4, 6), (5, 7), (6, 7)]
    xs, ys, zs = [], [], []
    for a, b in edges:
        xs += [c[a, 0], c[b, 0], None]; ys += [c[a, 1], c[b, 1], None]; zs += [c[a, 2], c[b, 2], None]
    return xs, ys, zs


centre_p = (cx + offset[0], cy + offset[1], cz + offset[2])
centre_win = min(range(len(windows)), key=lambda i: sum(
    ((w.start + w.stop) / 2 - c) ** 2 for w, c in zip(windows[i], centre_p)))

fig = go.Figure()
for mask, colour, opacity, name in ((brain, "lightgrey", 0.08, "brain"), (gt_regions[2], "#2ca02c", 0.5, "whole tumour")):
    m = mesh(mask, step=3 if name == "brain" else 1, sigma=2.0 if name == "brain" else 1.0)
    if m is not None:
        v, f = m
        fig.add_trace(go.Mesh3d(x=v[:, 0], y=v[:, 1], z=v[:, 2], i=f[:, 0], j=f[:, 1], k=f[:, 2],
                                color=colour, opacity=opacity, name=name, showlegend=True, hoverinfo="skip"))
xs, ys, zs = [], [], []
for i, w in enumerate(windows):
    if i == centre_win:
        continue
    bx, by, bz = box_lines(*(s.start - o for s, o in zip(w, offset)), *ROI)
    xs += bx; ys += by; zs += bz
fig.add_trace(go.Scatter3d(x=xs, y=ys, z=zs, mode="lines", line=dict(color="#5b7fb5", width=2),
                           name="windows", hoverinfo="skip"))
bx, by, bz = box_lines(*(s.start - o for s, o in zip(windows[centre_win], offset)), *ROI)
fig.add_trace(go.Scatter3d(x=bx, y=by, z=bz, mode="lines", line=dict(color="#d62728", width=6),
                           name="window with tumour centre", hoverinfo="skip"))
fig.update_layout(width=850, height=650, margin=dict(t=40, b=10, l=10, r=10),
                  scene=dict(aspectmode="data", camera=dict(eye=dict(x=1.6, y=-1.6, z=1.0)),
                             xaxis=dict(visible=False), yaxis=dict(visible=False), zaxis=dict(visible=False)),
                  legend=dict(orientation="h", x=0.5, xanchor="center", y=-0.02, yanchor="top"),
                  title=dict(text=f"<b>{len(windows)} windows of {ROI[0]}³</b>", x=0.5))
fig.show()

## 3. Gaussian blending weights

Predictions near a window's border are made with less context (zero padding inside the
network). When windows overlap, each window's output is therefore weighted by a 3D Gaussian:
high in the centre, low at the edges. Weights are summed per voxel and the weighted outputs
are divided by that sum.

In [35]:
importance = compute_importance_map(ROI, mode="gaussian", sigma_scale=0.125, device=DEVICE)
imp = importance.cpu().numpy()
mid = ROI[0] // 2

fig = make_subplots(rows=1, cols=2, subplot_titles=("weight, central slice of a window",
                                                    "weight along one axis through the centre"),
                    horizontal_spacing=0.12)
fig.add_trace(go.Heatmap(z=imp[:, :, mid].T, colorscale="Viridis", colorbar=dict(title="w", x=0.44)), 1, 1)
fig.add_trace(go.Scatter(y=imp[:, mid, mid], mode="lines", line=dict(color="#c56d6d", width=3),
                         showlegend=False), 1, 2)
fig.update_xaxes(visible=False, row=1, col=1)
fig.update_yaxes(visible=False, scaleanchor="x", row=1, col=1)
fig.update_xaxes(title="position in the window (voxels)", row=1, col=2)
fig.update_yaxes(title="weight", row=1, col=2)
fig.update_layout(width=1000, height=420, template="simple_white", margin=dict(t=50, b=50, l=20, r=20))
fig.show(config=STATIC)
print(f"weight at the centre: {imp.max():.3f}, at a face centre: {imp[0, mid, mid]:.4f}, at a corner: {imp[0, 0, 0]:.2e}")

weight at the centre: 0.999, at a face centre: 0.0010, at a corner: 1.00e-03


## 4. Inference as an explicit loop

The loop below is what `sliding_window_inference` does: run the model on every window,
multiply the output **logits** by the weights, add them into one volume, divide by the summed
weights, remove the padding. For a few chosen voxels, every window's contribution is recorded
(used in section 6). The result is compared with MONAI's function.

Probe voxels are picked from a reference prediction made with MONAI first.

In [36]:
ref_logits = predict_logits(model, image[None].to(DEVICE), ROI, sw_batch_size=4, overlap=OVERLAP, amp=AMP)[0].cpu()
ref_prob = torch.sigmoid(ref_logits).numpy()
ref_lab = regions_to_labels(ref_prob > 0.5)


def nearest_in(mask, point):
    idx = np.argwhere(mask)
    if not len(idx):
        return None
    return tuple(int(v) for v in idx[((idx - np.array(point)) ** 2).sum(1).argmin()])


rng = np.random.default_rng(SEED)
outside = np.argwhere(brain & ~gt_regions[2])
candidates = {
    "inside ET": nearest_in(gt_lab == 3, centroid(gt_lab == 3)),
    "ET border (p≈0.5)": nearest_in((np.abs(ref_prob[0] - 0.5) < 0.1) & gt_regions[2], centroid(gt_lab == 3)),
    "oedema (SNFH)": nearest_in(gt_lab == 2, centroid(gt_lab == 2)),
    "predicted NETC, not NETC in truth": nearest_in((ref_lab == 1) & (gt_lab != 1), centroid(gt_regions[2])),
    "healthy brain": tuple(int(v) for v in outside[rng.integers(len(outside))]) if len(outside) else None,
}
probes = {k: v for k, v in candidates.items() if v is not None}
pd.DataFrame({"voxel (x, y, z)": [str(v) for v in probes.values()],
              "true label": [LABEL_NAMES[int(gt_lab[v])] for v in probes.values()]}, index=list(probes))

,"voxel (x, y, z)",true label
inside ET,"(28, 27, 98)",ET
ET border (p≈0.5),"(25, 25, 101)",ET
oedema (SNFH),"(35, 38, 87)",SNFH
"predicted NETC, not NETC in truth","(31, 32, 92)",background
healthy brain,"(108, 58, 10)",background


In [37]:
n_ch = label.shape[0]
acc = torch.zeros((n_ch, *padded), device=DEVICE)      # weighted sum of logits
wsum = torch.zeros(padded, device=DEVICE)              # sum of weights
cover = torch.zeros(padded, device=DEVICE)             # number of windows per voxel
records = {name: [] for name in probes}
img_p = image_p.to(DEVICE)

with torch.no_grad(), torch.autocast(device_type=DEVICE.type, dtype=torch.bfloat16, enabled=AMP):
    for w_idx, (sx, sy, sz) in enumerate(windows):
        out = model(img_p[None, :, sx, sy, sz])[0].float()   # (C, 128, 128, 128) logits
        acc[:, sx, sy, sz] += out * importance
        wsum[sx, sy, sz] += importance
        cover[sx, sy, sz] += 1
        for name, (x, y, z) in probes.items():
            px, py, pz = x + offset[0], y + offset[1], z + offset[2]
            if sx.start <= px < sx.stop and sy.start <= py < sy.stop and sz.start <= pz < sz.stop:
                lx, ly, lz = px - sx.start, py - sy.start, pz - sz.start
                records[name].append({"window": w_idx, "weight": importance[lx, ly, lz].item(),
                                      **{r: out[c, lx, ly, lz].item() for c, r in enumerate(regions)}})

crop = tuple(slice(o, o + s) for o, s in zip(offset, size))
manual_logits = (acc / wsum)[(slice(None), *crop)].cpu()
cover = cover[crop].cpu().numpy()
wsum = wsum[crop].cpu().numpy()

diff = (torch.sigmoid(manual_logits) - torch.sigmoid(ref_logits)).abs()
same_mask = ((manual_logits > 0) == (ref_logits > 0)).float().mean().item()
print(f"max |probability difference| vs MONAI: {diff.max().item():.2e}")
print(f"voxels with the same thresholded decision: {100 * same_mask:.4f} %")

max |probability difference| vs MONAI: 4.77e-07
voxels with the same thresholded decision: 100.0000 %


## 5. One window, and how many windows cover each voxel

Left: the window containing the tumour centre, its raw ET probability (this window alone),
its weights, and the final blended ET probability on the same area. Below: the number of
windows covering each voxel of the slice, and the summed Gaussian weight.

In [38]:
sx, sy, sz = windows[centre_win]
with torch.no_grad(), torch.autocast(device_type=DEVICE.type, dtype=torch.bfloat16, enabled=AMP):
    win_prob = torch.sigmoid(model(img_p[None, :, sx, sy, sz])[0].float()).cpu().numpy()

lz = cz + offset[2] - sz.start           # tumour-centre slice inside the window
xs_, ys_ = slice(sx.start - offset[0], sx.stop - offset[0]), slice(sy.start - offset[1], sy.stop - offset[1])
final_et = torch.sigmoid(manual_logits[0]).numpy()
final_crop = np.full(ROI[:2], np.nan)
xa, xb = max(xs_.start, 0), min(xs_.stop, size[0])
ya, yb = max(ys_.start, 0), min(ys_.stop, size[1])
final_crop[xa - xs_.start:xb - xs_.start, ya - ys_.start:yb - ys_.start] = final_et[xa:xb, ya:yb, cz]

diff_crop = final_crop - win_prob[0][:, :, lz]   # NaN outside the volume stays NaN

panels = [
    ("t1c in the window", image_p[1][sx, sy, sz].numpy()[:, :, lz], "gray", None),
    ("ET probability, this window only", win_prob[0][:, :, lz], "Viridis", (0, 1)),
    ("Gaussian weight", imp[:, :, lz], "Viridis", None),
    ("ET probability, blended", final_crop, "Viridis", (0, 1)),
    ("blended − this window", diff_crop, "RdBu", (-1, 1)),
]
fig = make_subplots(rows=1, cols=5, subplot_titles=[p[0] for p in panels], horizontal_spacing=0.015)
for i, (_, z2d, cs, zr) in enumerate(panels, start=1):
    fig.add_trace(go.Heatmap(z=z2d.T, colorscale=cs, showscale=i == 5, hoverinfo="skip",
                             zmin=zr[0] if zr else None, zmax=zr[1] if zr else None,
                             zmid=0 if cs == "RdBu" else None,
                             colorbar=dict(title="Δp", len=0.8) if i == 5 else None), 1, i)
fig.update_xaxes(visible=False)
fig.update_yaxes(visible=False)
for i in range(1, 6):
    sx_ = "" if i == 1 else str(i)
    fig.layout[f"yaxis{sx_}"].update(scaleanchor=f"x{sx_}")
fig.update_layout(width=1400, height=340, margin=dict(t=40, b=10, l=10, r=10))
fig.show(config=STATIC)
print(f"max |blended − this window| on this slice: {np.nanmax(np.abs(diff_crop)):.3f}")

fig = make_subplots(rows=1, cols=2, subplot_titles=(f"windows covering each voxel (z = {cz})",
                                                    "sum of Gaussian weights"), horizontal_spacing=0.12)
fig.add_trace(go.Heatmap(z=xy_slice(cover, cz), colorscale="Blues", colorbar=dict(title="n", x=0.44)), 1, 1)
fig.add_trace(go.Heatmap(z=xy_slice(wsum, cz), colorscale="Viridis", colorbar=dict(title="Σw")), 1, 2)
fig.update_xaxes(visible=False)
fig.update_yaxes(visible=False)
fig.layout.yaxis.update(scaleanchor="x")
fig.layout.yaxis2.update(scaleanchor="x2")
fig.update_layout(width=950, height=430, margin=dict(t=40, b=10, l=10, r=10))
fig.show(config=STATIC)
print(f"windows per voxel: min {int(cover.min())}, max {int(cover.max())}")

max |blended − this window| on this slice: 0.207


windows per voxel: min 1, max 8


## 6. How one voxel gets its label

For a voxel $v$, every window $k$ that contains it gives one logit per region
(ET, TC, WT, RC). The final label is decided in five steps:

1. **Window outputs.** Each window $k$ containing $v$ gives $\text{logit}_k(v)$
   (1 to 8 windows, depending on where $v$ lies).
2. **Weighted mean of logits.** Each window's logit is weighted by the Gaussian weight
   $w_k(v)$ of the voxel's position in that window (high near the window centre, low near
   its edge), then normalised by the sum of weights:

   $$\text{logit}(v) = \frac{\sum_k w_k(v)\,\text{logit}_k(v)}{\sum_k w_k(v)}$$

   Only the ratios of the weights matter. Logits are averaged, not probabilities.
3. **Sigmoid per region.** $p = \sigma(\text{logit})$, separately for ET, TC, WT and RC.
4. **Threshold per region.** Each region is on if $p > 0.5$. Regions are independent,
   so several can be on at once.
5. **Nesting rule.** The innermost region that is on gives the label:
   ET → 3, otherwise TC → 1 (NETC), otherwise WT → 2 (SNFH), otherwise RC → 4,
   otherwise 0 (background).

The label is **not** the region with the highest probability. Example:

| region | $p$ | $p > 0.5$ |
|---|---|---|
| ET | 0.81 | yes |
| TC | 0.98 | yes |
| WT | 0.93 | yes |
| RC | 0.05 | no |

ET is on, so the voxel is labelled **ET (3)**. Taking the highest probability would give TC,
which without ET means NETC, a different class.

The table below applies these steps to a few chosen voxels of this case.

In [47]:
rows = []
manual_prob = torch.sigmoid(manual_logits).numpy()
for name, v in probes.items():
    p = manual_prob[(slice(None), *v)]
    row = {"probe": name, "true label": LABEL_NAMES[int(gt_lab[v])], "windows": len(records[name])}
    for c, r in enumerate(regions):
        row[f"p({r})"] = round(float(p[c]), 3)
    row["predicted label"] = LABEL_NAMES[int(regions_to_labels((p > 0.5)[:, None])[0])]
    rows.append(row)
pd.DataFrame(rows).set_index("probe")

,true label,windows,p(ET),p(TC),p(WT),p(RC),predicted label
probe,,,,,,,
inside ET,ET,4,0.999,1.000,0.996,0.001,ET
ET border (p≈0.5),ET,4,0.439,0.562,0.351,0.006,NETC
oedema (SNFH),SNFH,4,0.000,0.000,0.996,0.008,SNFH
"predicted NETC, not NETC in truth",background,4,0.342,0.558,0.399,0.002,NETC
healthy brain,background,4,0.000,0.000,0.000,0.000,background


In [40]:
PROBE = "ET border (p≈0.5)" if "ET border (p≈0.5)" in records else next(iter(records))
REGION = "ET"
c = regions.index(REGION)

t = pd.DataFrame(records[PROBE])
t[f"p({REGION}) this window"] = 1 / (1 + np.exp(-t[REGION]))
t["share of weight"] = t["weight"] / t["weight"].sum()
blended_logit = (t[REGION] * t["weight"]).sum() / t["weight"].sum()

print(f"{PROBE} at {probes[PROBE]}, region {REGION}")
print(f"blended logit = Σ(w·logit) / Σw = {blended_logit:.3f}  ->  p = {1 / (1 + np.exp(-blended_logit)):.3f}"
      f"  ->  {'inside' if blended_logit > 0 else 'outside'} {REGION}")
t[["window", "weight", "share of weight", REGION, f"p({REGION}) this window"]].rename(
    columns={REGION: f"logit {REGION}"}).round(4).set_index("window")

ET border (p≈0.5) at (25, 25, 101), region ET
blended logit = Σ(w·logit) / Σw = -0.247  ->  p = 0.439  ->  outside ET


,weight,share of weight,logit ET,p(ET) this window
window,,,,
0,0.001,0.25,-1.4987,0.1826
1,0.001,0.25,0.0350,0.5087
4,0.001,0.25,-0.8113,0.3076
5,0.001,0.25,1.2862,0.7835


### Blending for one voxel

All windows that contain the chosen voxel, with the weight of the voxel's position inside
each window and that window's own ET probability. The blended logit is the weighted mean
of the window logits; the final probability is its sigmoid (not the mean of probabilities).

In [ ]:
HALF = 48  # field of view: 96 × 96 voxels around the tumour centre
x0, x1 = max(cx - HALF, 0), min(cx + HALF, size[0])
y0, y1 = max(cy - HALF, 0), min(cy + HALF, size[1])

# windows containing the tumour centre (padded coordinates)
pc = (cx + offset[0], cy + offset[1], cz + offset[2])
covering = [i for i, w in enumerate(windows) if all(s.start <= p < s.stop for s, p in zip(w, pc))]

probs_fov, weights_fov, titles = [], [], []
with torch.no_grad(), torch.autocast(device_type=DEVICE.type, dtype=torch.bfloat16, enabled=AMP):
    for i in covering:
        sx, sy, sz = windows[i]

        p_win = torch.sigmoid(model(img_p[None, :, sx, sy, sz])[0, 0].float()).cpu().numpy()
        lz = pc[2] - sz.start

        prob_g = np.full(size[:2], np.nan)
        w_g = np.full(size[:2], np.nan)
        # window region in original (unpadded) coordinates, clipped to the volume
        gx0, gx1 = max(sx.start - offset[0], 0), min(sx.stop - offset[0], size[0])
        gy0, gy1 = max(sy.start - offset[1], 0), min(sy.stop - offset[1], size[1])

        lx0, ly0 = gx0 + offset[0] - sx.start, gy0 + offset[1] - sy.start

        prob_g[gx0:gx1, gy0:gy1] = p_win[lx0:lx0 + gx1 - gx0, ly0:ly0 + gy1 - gy0, lz]
        w_g[gx0:gx1, gy0:gy1] = imp[lx0:lx0 + gx1 - gx0, ly0:ly0 + gy1 - gy0, lz]

        probs_fov.append(prob_g[x0:x1, y0:y1])
        weights_fov.append(w_g[x0:x1, y0:y1])
        
        w_centre = imp[pc[0] - sx.start, pc[1] - sy.start, lz]

        titles.append(f"window {i}<br>weight here: {w_centre:.2f}")

blended = torch.sigmoid(manual_logits[0]).numpy()[x0:x1, y0:y1, cz]
weight_sum = wsum[x0:x1, y0:y1, cz]
truth = gt_regions[0][x0:x1, y0:y1, cz].astype(float)

n = len(covering) + 1
fig = make_subplots(rows=2, cols=n, horizontal_spacing=0.01, vertical_spacing=0.08,
                    subplot_titles=titles + ["<b>blended</b>"] + [""] * (n - 1) + ["<b>sum of weights</b>"])

for col, (p, w) in enumerate(zip(probs_fov + [blended], weights_fov + [weight_sum]), start=1):
    fig.add_trace(go.Contour(z=truth.T, contours=dict(start=0.5, end=0.5, coloring="none"),
                             line=dict(color="white", width=1.5), showscale=False, showlegend=False,
                             hoverinfo="skip"), 1, col)

    fig.add_trace(go.Heatmap(z=p.T, zmin=0, zmax=1, colorscale="Viridis", showscale=col == n,
                             colorbar=dict(title="p(ET)", y=0.78, len=0.42), hoverinfo="skip"), 1, col)

    fig.add_trace(go.Contour(z=truth.T, contours=dict(start=0.5, end=0.5, coloring="none"),
                             line=dict(color="white", width=1.5), showscale=False, hoverinfo="skip"), 1, col)

    fig.add_trace(go.Heatmap(z=w.T, zmin=0, zmax=float(np.nanmax(weight_sum)), colorscale="Magma",
                             showscale=col == n, colorbar=dict(title="w", y=0.22, len=0.42),
                             hoverinfo="skip"), 2, col)

fig.update_xaxes(visible=False)
fig.update_yaxes(visible=False)

for i in range(1, 2 * n + 1):
    sx_ = "" if i == 1 else str(i)
    fig.layout[f"yaxis{sx_}"].update(scaleanchor=f"x{sx_}")

n_true_et = int(gt_regions[0][:, :, cz].sum())
contour_note = "white contour: true ET" if n_true_et else "no true ET on this slice — all predicted ET is false positive"
    
fig.update_layout(width=230 * n + 80, height=560, plot_bgcolor="#e5e5e5", showlegend=False, 
                  margin=dict(t=110, b=10, l=10, r=10),
                  title=dict(text=f"<b>Slice z = {cz}</b>: {len(covering)} windows contain the tumour centre "
                                  f"({contour_note})", x=0.5, y=0.98))
fig.show(config=STATIC)